# 01 · 样本观察：一天的广告竞价日志长什么样

本系列 notebook 带你从零理解 AuctionNet 竞价市场，并亲手实现 PID / IQL / DT / LLM 四种竞价策略。

**本篇任务**：读懂官方数据集的 18 列 schema，观察一个广告主一整天（48 个时段、50 万次展示机会）的竞价过程。

数据：`data/period7_adv0.csv.gz` = 官方 period-7 数据中 0 号广告主的全部日志（本仓库自带，18.8MB）。

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caitq2024/auto_auction/blob/main/notebooks/01_%E6%A0%B7%E6%9C%AC%E8%A7%82%E5%AF%9F.ipynb)

> Colab 用户先运行下面的数据下载 cell；本地运行可跳过。


In [ ]:
# Colab 环境准备：拉取教学数据（本地运行且 data/ 已存在时自动跳过）
import os, urllib.request
os.makedirs('data', exist_ok=True)
for f in ['period7_adv0.csv.gz', 'period7_tick0_adv0to7.csv.gz']:
    if not os.path.exists(f'data/{f}'):
        urllib.request.urlretrieve(f'https://github.com/caitq2024/auto_auction/raw/main/notebooks/data/{f}', f'data/{f}')
        print('downloaded', f)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/period7_adv0.csv.gz')
print(f'行数: {len(df):,}  (= 该广告主一天参与竞价的展示机会数)')
df.head(3).T

## 关键列速查

| 列 | 含义 |
|---|---|
| `pValue` | 平台预估的转化概率（该次展示给这个广告主带来转化的概率）|
| `bid` | 该广告主的出价 = 当时的系数 alpha × pValue |
| `leastWinningCost` | 该次展示的最低获胜价（第 4 高出价，进前三的门槛）|
| `xi / adSlot / isExposed` | 是否赢得坑位 / 坑位号 1-3 / 是否实际曝光 |
| `cost` | GSP 扣费（只有曝光了才真扣钱）|
| `conversionAction` | 是否发生转化（按 pValue 抽样的结果）|

In [ ]:
# 该广告主的预算与约束
print('budget:', df.budget.iloc[0], ' target CPA:', df.CPAConstraint.iloc[0])

# 一天的战绩汇总
total_cost = df.cost[df.isExposed == 1].sum()
total_conv = df.conversionAction.sum()
print(f'花费 {total_cost:.0f} / 转化 {total_conv:.0f} / 实际CPA {total_cost/max(total_conv,1):.1f}')

In [ ]:
# 时段结构：流量、价格、赢单率如何随一天变化
by_tick = df.groupby('timeStepIndex').agg(
    pv=('pvIndex', 'count'),
    pvalue_mean=('pValue', 'mean'),
    market_price=('leastWinningCost', 'mean'),
    win_rate=('xi', 'mean'),
    spend=('cost', lambda c: c[df.loc[c.index, 'isExposed'] == 1].sum()),
)
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
by_tick.pv.plot(ax=axes[0,0], title='每时段展示机会数'); 
by_tick.market_price.plot(ax=axes[0,1], title='市场价（最低获胜价均值）')
by_tick.win_rate.plot(ax=axes[1,0], title='赢单率')
by_tick.spend.plot(ax=axes[1,1], title='每时段花费')
plt.tight_layout()

## 观察要点（思考题）

1. 市场价一天内波动多大？最贵和最便宜的时段差几倍？
2. 这个广告主的钱是在哪些时段花掉的？它"错过"便宜时段了吗？
3. `bid / pValue` 反推出它每个时段的系数 alpha——它是怎么调的？（下一篇实现 PID 时对照）

In [ ]:
# 反推有效 alpha 序列
alpha_seq = (df.groupby('timeStepIndex')
               .apply(lambda g: g.bid.mean() / g.pValue.mean(), include_groups=False))
alpha_seq.plot(figsize=(10, 3), title='该广告主的有效 alpha 序列（bid_mean / pValue_mean）')